In [13]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict,Annotated
from dotenv import load_dotenv
import os
from pydantic import BaseModel, Field
import operator


In [2]:
load_dotenv()

True

In [7]:
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key="YOUR_API_KEY" 
)

In [4]:
class EvaluationSchema(BaseModel):

    feedback: str = Field(description="Detailed feedback for the essay")
    score: int = Field(description="Score out of 10",ge=0,le=10)

In [8]:
structured_model = model.with_structured_output(EvaluationSchema)

In [9]:
essay = """# **Technology as the Silent Architect of Human Civilization**

## Introduction

Technology has always been the invisible force shaping human civilization. From the invention of the wheel to the rise of Artificial Intelligence, every technological breakthrough has transformed the way humans live, work, communicate, and govern themselves. It is often said that "Technology is a useful servant but a dangerous master." This statement highlights the dual nature of technology—it can empower humanity or create new challenges depending on how it is used.

Today, technology is no longer confined to laboratories or industries. It influences education, healthcare, agriculture, governance, business, national security, and even personal relationships. Therefore, understanding its opportunities and limitations is essential for building a sustainable and inclusive future.

## Technology and Economic Development

Technology acts as the engine of economic growth. Industrial revolutions transformed agrarian economies into manufacturing giants, while the digital revolution created entirely new industries such as software development, e-commerce, fintech, and artificial intelligence.

Countries investing in innovation generally experience higher productivity, better employment opportunities, and improved standards of living. Digital payment systems, online banking, automation, and cloud computing have increased efficiency while reducing transaction costs.

For a developing country like India, initiatives such as Digital India, Startup India, and Make in India demonstrate how technology can accelerate inclusive development.

## Technology in Education

Education has witnessed revolutionary changes through technology. Online learning platforms, digital classrooms, virtual laboratories, and AI-powered learning assistants have made education more accessible than ever.

Students living in remote villages can now access lectures from leading universities across the world. During the COVID-19 pandemic, digital education ensured continuity of learning despite physical restrictions.

However, unequal internet access, lack of digital infrastructure, and varying levels of digital literacy continue to create a digital divide. Technology should complement teachers rather than replace them, ensuring that education remains both accessible and human-centered.

## Technology in Healthcare

Modern healthcare has been transformed by scientific innovation. Artificial Intelligence assists doctors in diagnosing diseases, robotic surgery improves precision, telemedicine connects specialists with rural patients, and wearable devices enable continuous health monitoring.

India's digital health initiatives and vaccination management systems demonstrated how technology can improve healthcare delivery on a massive scale.

Nevertheless, concerns regarding patient privacy, cybersecurity, affordability, and ethical use of medical data require strong regulatory frameworks.

## Technology and Governance

Good governance depends upon transparency, accountability, and efficient service delivery. Digital governance has reduced corruption, minimized paperwork, and improved accessibility to government services.

Online tax filing, digital identity systems, direct benefit transfers, land record digitization, and e-governance portals have significantly improved administrative efficiency.

Technology also strengthens democracy by increasing citizen participation through digital platforms. However, governments must ensure cybersecurity, data protection, and equal digital access to prevent exclusion.

## Artificial Intelligence: Opportunities and Challenges

Artificial Intelligence represents one of the most transformative technologies of the 21st century. AI enhances productivity, automates repetitive tasks, assists scientific research, and supports decision-making in various sectors.

Its applications range from agriculture and healthcare to finance, transportation, and disaster management.

Yet AI also raises significant concerns:

* Job displacement due to automation.
* Algorithmic bias and discrimination.
* Privacy violations.
* Spread of misinformation.
* Ethical dilemmas regarding autonomous systems.

Therefore, AI development must be guided by ethical principles emphasizing transparency, accountability, fairness, and human oversight.

## Environmental Dimension

Technology has both contributed to environmental degradation and offered solutions for sustainability.

Industrialization increased pollution and carbon emissions, but technological innovation now supports renewable energy, electric vehicles, smart grids, precision agriculture, and waste management.

Climate change requires technological solutions combined with responsible consumption and sustainable policies. Green technology represents the future of balanced development.

## Ethical Concerns

Rapid technological advancement raises important ethical questions.

Should machines make life-and-death decisions?

How should personal data be protected?

Who owns data generated by individuals?

Can technological progress justify increasing inequality?

Ethics must evolve alongside innovation. Responsible innovation demands legal safeguards, transparency, public accountability, and respect for human dignity.

## India's Technological Journey

India has emerged as a global technology leader through its IT industry, digital public infrastructure, space missions, pharmaceutical innovation, and startup ecosystem.

Achievements such as UPI, Aadhaar-enabled services, Chandrayaan missions, digital governance, and expanding internet connectivity illustrate India's technological capabilities.

However, challenges remain:

* Digital divide.
* Cybersecurity threats.
* Skill gaps.
* Research investment.
* Rural digital infrastructure.
* Data privacy concerns.

Addressing these issues will enable India to become a technology-driven knowledge economy.

## The Way Forward

To maximize technological benefits while minimizing risks, several steps are necessary:

* Invest in research and innovation.
* Strengthen digital infrastructure.
* Promote ethical AI frameworks.
* Improve cybersecurity.
* Bridge the rural-urban digital divide.
* Enhance digital literacy.
* Encourage public-private partnerships.
* Ensure inclusive technological development.

Technology should empower every citizen regardless of geography, income, or social background.

## Conclusion

Technology is neither inherently good nor bad; its impact depends on human values, policies, and intentions. Throughout history, technological progress has expanded human capabilities, improved living standards, and solved complex problems. At the same time, it has introduced new ethical, social, and environmental challenges.

The true measure of technological advancement lies not merely in innovation but in its ability to promote justice, equality, sustainability, and human welfare. As India aspires to become a developed nation, technology must remain a tool for inclusive growth rather than an instrument of exclusion.

Ultimately, the future belongs not to those who simply invent technology, but to those who use it wisely, ethically, and for the benefit of humanity.
"""

In [ ]:
prompt = f'Evalute the language quality of the following essay and provide a feedback and assign a score out of 10 \n {essay}'
structured_model.invoke(prompt)

In [19]:
class UPSCState(TypedDict):

    essay:str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores:Annotated[list[int],operator.add]
    avg_score:float

In [20]:
def evaluate_language(state:UPSCState):

    prompt=f'Evalute the language quality of the following essay and provide a feedback and assign a score out of 10 \n {essay}'
    output=structured_model.invoke(prompt)

    return {'language_feedback':output.feedback, 'indiviual_scores':[output.score]}

In [21]:
def evaluate_analysis(state:UPSCState):

    prompt=f'Evalute the depth of the analysis of the following essay and provide a feedback and assign a score out of 10 \n {essay}'
    output=structured_model.invoke(prompt)

    return {'analysis_feedback':output.feedback, 'indiviual_scores':[output.score]}

In [22]:
def evaluate_thought(state:UPSCState):

    prompt=f'Evalute the clarity of thought of the analysis of the following essay and provide a feedback and assign a score out of 10 \n {essay}'
    output=structured_model.invoke(prompt)

    return {'clarity_feedback':output.feedback, 'indiviual_scores':[output.score]}

In [23]:
def find_evaluation(state:UPSCState):

   prompt=f''

   overall_feedback = model.invoke(prompt)

   avg_score=sum(state['individual_scores'])/len(state['individual_scores'])

   return {'overall_feedback':overall_feedback, 'avg_score':avg_score}

In [ ]:
graph = StateGraph(UPSCState)

graph.add_node("evaluate_language",evaluate_language)
graph.add_node("evaluate_analysis",evaluate_analysis)
graph.add_node("evaluate_thought",evaluate_thought)
graph.add_node("find_evaluation",find_evaluation)


graph.add_edge(START,'evaluate_language')
graph.add_edge(START,'evaluate_analysis')
graph.add_edge(START,'evaluate_thought')

graph.add_edge('evaluate_language',find_evaluation)
graph.add_edge('evaluate_analysis',find_evaluation)
graph.add_edge('evaluate_thought',find_evaluation)

graph.add_edge('find_evaluation',END)

workflow=graph.compile()

In [ ]:
initial_state={
    'essay':essay
}
workflow.invoke(initial_state)